In [1]:
import importlib
import sys
import json
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display, HTML, Markdown

# -------------------- Dark Theme --------------------
display(HTML("""
<style>
    body, .jp-Notebook, .jp-OutputArea-output, .jp-RenderedHTMLCommon {
        background-color: #1e1e1e !important;
        color: #d4d4d4 !important;
    }
    h2, h3, h4 {
        color: #4fc3f7 !important;
        border-bottom: 2px solid #3498db !important;
        padding-bottom: 4px;
    }
    b, strong { color: #f48fb1 !important; }
    .highlight {
        background-color: #2d2d2d !important;
        padding: 10px;
        border-left: 4px solid #3498db;
        margin: 4px 0;
        color: #d4d4d4;
    }
    code {
        background-color: #333 !important;
        color: #ffcc80 !important;
        padding: 2px 4px;
        border-radius: 4px;
    }
    .dataframe {
        background-color: #2d2d2d !important;
        color: #d4d4d4 !important;
    }
</style>
"""))

# -------------------- Verbosity Flags --------------------
SHOW_VERBOSE = True
SHOW_INFO = True
SHOW_CRITICAL = True
SHOW_DEBUG = True

# -------------------- Helper functions for display --------------------
def display_title(title: str):
    display(HTML(f"<h2>{title}</h2>"))

def display_info(message: str):
    display(HTML(f"<div class='highlight'>{message}</div>"))

print("Environment ready. Dark theme applied.")

Environment ready. Dark theme applied.


In [2]:
# =============================================================================
# CELL 2 – LOAD AND INSPECT DATASETS
# =============================================================================

from Get_Go_Emo import get_go
from Get_Isear import get_isr

go_df = get_go()
isear_df = get_isr()

DATASETS = {
    "goEmo": go_df,
    "ISEAR": isear_df,
}

for name, df in DATASETS.items():
    display_title(f"Dataset: {name}")
    display_info(f"Shape: {df.shape}")
    display(df.head(2))
    # We'll rely on the probe's column detection later
    display_info(f"Samples: <b>{len(df):,}</b>")

display_title("Unified ID Scheme")
display_info("""
Each sample is assigned a unique integer ID (0..N-1) matching its row index.
This ID links:
• Input text (original DataFrame index)
• Hidden state vector (row in hidden_states.npy)
• Label (row in labels.npy)
No shuffling occurs, guaranteeing one‑to‑one mapping.
""")

,labels,clean_text
0,[27],my favourite food is anything i didnt have to ...
1,[27],"now if he does off himself, everyone will thin..."


,clean_text,labels
0,during the period of falling in love each time...,1
1,when i was involved in a traffic accident,2


In [3]:
# =============================================================================
# CELL 2 – LOAD AND INSPECT DATASETS
# =============================================================================

from Get_Go_Emo import get_go
from Get_Isear import get_isr

goEmo = get_go()
isear = get_isr()

DATASETS = {
    "goEmo": goEmo,
    "ISEAR": isear,
}

display_title("Unified ID Scheme")
display_info("""
Each sample is assigned a unique integer ID (0..N-1) matching its row index.
This ID links:
• Input text (original DataFrame index)
• Hidden state vector (row in hidden_states.npy)
• Label (row in labels.npy)
No shuffling occurs, guaranteeing one‑to‑one mapping.
""")

In [4]:
import unified_hidden_state_probe_v4_2 as probe

GOEMOTIONS_CLASSES = probe.GOEMOTIONS_CLASSES
ISEAR_CLASSES = probe.ISEAR_CLASSES
EXTERNAL_ROOT = Path('/Volumes/Amirali/hidden_states')
EXPERIMENT_ID = 'baseline_v5_001'

# Contracts with auto column detection and lenient provenance
goemotions_contract = probe.DatasetContract(
    target_type='goemotions',
    text_column='auto',
    label_column='auto',
    id_column='auto',
    task_type='multi_label',
    class_order=GOEMOTIONS_CLASSES,
    lenient_provenance=True,      # allow head/tail match
    require_provenance=False,
)

isear_contract = probe.DatasetContract(
    target_type='isear',
    text_column='auto',
    label_column='auto',
    id_column='auto',
    task_type='single_label',
    class_order=ISEAR_CLASSES,
    lenient_provenance=True,
    require_provenance=False,
)

# Probes definition (same as before)
probes = [
    probe.ProbeSpec(name='linear_logistic', type='logistic', complexity='linear',
                    standardize=True, C=1.0, max_iter=3000, selection_metric='macro_f1'),
    probe.ProbeSpec(name='mlp_1_hidden', type='mlp', complexity='1_hidden',
                    standardize=True, hidden_dims=['0.5d'], learning_rate=1e-3,
                    weight_decay=1e-4, epochs=80, batch_size=256, patience=12,
                    selection_metric='macro_f1'),
    probe.ProbeSpec(name='mlp_2_hidden', type='mlp', complexity='2_hidden',
                    standardize=True, hidden_dims=['0.5d', '0.25d'], learning_rate=1e-3,
                    weight_decay=1e-4, epochs=80, batch_size=256, patience=12,
                    selection_metric='macro_f1'),
    probe.ProbeSpec(name='mlp_3_hidden', type='mlp', complexity='3_hidden',
                    standardize=True, hidden_dims=['0.5d', '0.25d', '0.125d'], learning_rate=1e-3,
                    weight_decay=1e-4, epochs=80, batch_size=256, patience=12,
                    selection_metric='macro_f1'),
]
print('Contracts and probes defined (auto columns, lenient provenance).')

Contracts and probes defined (auto columns, lenient provenance).


In [5]:
all_pairs = probe.discover_model_dataset_pairs(EXTERNAL_ROOT, EXPERIMENT_ID)
print(f"Found {len(all_pairs)} model-dataset pairs.")
display(pd.DataFrame(all_pairs))

Found 17 model-dataset pairs.


,model,dataset
0,google-bert/bert-base-uncased,goEmo
1,google-bert/bert-base-uncased,ISEAR
2,distilbert/distilbert-base-uncased,goEmo
3,distilbert/distilbert-base-uncased,ISEAR
4,FacebookAI/roberta-base,goEmo
5,FacebookAI/roberta-base,ISEAR
6,google/electra-small-discriminator,goEmo
7,google/electra-small-discriminator,ISEAR
8,microsoft/deberta-v3-small,goEmo
9,microsoft/deberta-v3-small,ISEAR


In [6]:
dataset_map = {
    'goEmo': (goemotions_contract, go_df),
    'ISEAR': (isear_contract, isear_df),
}

entries = []
for pair in all_pairs:
    model = pair['model']
    dataset = pair['dataset']
    contract, df = dataset_map.get(dataset, (None, None))
    if contract is None:
        continue
    entries.append({
        'model': model,
        'dataset': dataset,
        'contract': contract,
        'dataset_df': df,
    })

print(f"Prepared {len(entries)} matrix entries.")
display(pd.DataFrame(entries)[['model', 'dataset']].head(10))

Prepared 17 matrix entries.


,model,dataset
0,google-bert/bert-base-uncased,goEmo
1,google-bert/bert-base-uncased,ISEAR
2,distilbert/distilbert-base-uncased,goEmo
3,distilbert/distilbert-base-uncased,ISEAR
4,FacebookAI/roberta-base,goEmo
5,FacebookAI/roberta-base,ISEAR
6,google/electra-small-discriminator,goEmo
7,google/electra-small-discriminator,ISEAR
8,microsoft/deberta-v3-small,goEmo
9,microsoft/deberta-v3-small,ISEAR


In [7]:
VERBOSE = 1
MAX_SAMPLES = 500
REPEATS = 1

checkpoint_dir = EXTERNAL_ROOT / 'experiments' / EXPERIMENT_ID / 'matrix_checkpoint'

full_results = probe.run_matrix(
    entries,
    external_root=EXTERNAL_ROOT,
    experiment_id=EXPERIMENT_ID,
    probes=probes,
    repeats=REPEATS,
    max_samples=MAX_SAMPLES,
    verbose=VERBOSE,
    checkpoint_dir=checkpoint_dir, 
)


[matrix] 1/17 | google-bert/bert-base-uncased | goEmo
[probe +    0.00s] ================================================================================================
[probe +    0.00s] INITIALISING UNIFIED HIDDEN-STATE PROBE
[probe +    0.00s] ================================================================================================
[probe +    4.03s] Model: google-bert/bert-base-uncased
[probe +    4.03s] Dataset artifact: goEmo
[probe +    4.03s] Hidden-state shape: (54263, 13, 768)
[probe +    4.03s] Task type: multi_label | classes: 28
[probe +    4.03s] Selected layers: 13 | device: cpu
[probe +    4.03s] Alignment: text=verified | labels=unverified
[probe +    4.12s] ================================================================================================
[probe +    4.12s] PROBING EXPERIMENT
[probe +    4.12s] ================================================================================================
[probe +    4.12s] Question: how recoverable is the targe

Probing:   0%|          | 0/208 [00:00<?, ?fit/s]

[probe + 2680.08s] Complete run metadata saved: /Volumes/Amirali/hidden_states/experiments/baseline_v5_001/models/google-bert/bert-base-uncased/datasets/goEmo/analysis/probes/matrix_runs/unified_v4_20260829_113941/complete_run_metadata.json
[probe + 2680.10s] ================================================================================================
[probe + 2680.10s] FINAL RESULT
[probe + 2680.10s] ================================================================================================
[probe + 2680.10s] Final best layer table:
          probe  layer_index  relative_layer_depth  probe_score_mean  test_macro_f1_mean  test_balanced_accuracy_mean  test_mcc_mean  selectivity_mean
linear_logistic            1              0.083333          0.761901            0.678571                     0.678571       0.981569          0.674099
   mlp_1_hidden            3              0.250000          0.731322            0.642857                     0.642857       0.972243          0.614091

Probing:   0%|          | 0/112 [00:00<?, ?fit/s]

[probe + 1018.05s] Complete run metadata saved: /Volumes/Amirali/hidden_states/experiments/baseline_v5_001/models/distilbert/distilbert-base-uncased/datasets/goEmo/analysis/probes/matrix_runs/unified_v4_20260829_122423/complete_run_metadata.json
[probe + 1018.10s] ================================================================================================
[probe + 1018.10s] FINAL RESULT
[probe + 1018.10s] ================================================================================================
[probe + 1018.10s] Final best layer table:
          probe  layer_index  relative_layer_depth  probe_score_mean  test_macro_f1_mean  test_balanced_accuracy_mean  test_mcc_mean  selectivity_mean
linear_logistic            5              0.833333          0.346438            0.113783                     0.101190       0.286477          0.096704
   mlp_1_hidden            3              0.500000          0.314559            0.105530                     0.193452       0.133689          0.0

...

Probing:   0%|          | 0/112 [00:00<?, ?fit/s]

[probe +  717.43s] Complete run metadata saved: /Volumes/Amirali/hidden_states/experiments/baseline_v5_001/models/microsoft/deberta-v3-small/datasets/goEmo/analysis/probes/matrix_runs/unified_v4_20260829_134715/complete_run_metadata.json
[probe +  717.45s] ================================================================================================
[probe +  717.45s] FINAL RESULT
[probe +  717.45s] ================================================================================================
[probe +  717.45s] Final best layer table:
          probe  layer_index  relative_layer_depth  probe_score_mean  test_macro_f1_mean  test_balanced_accuracy_mean  test_mcc_mean  selectivity_mean
linear_logistic            4              0.666667          0.358722            0.116667                     0.105357       0.409449          0.091030
   mlp_1_hidden            4              0.666667          0.324978            0.120199                     0.203571       0.218113          0.050781
  

Probing:   0%|          | 0/208 [00:00<?, ?fit/s]

[probe + 1407.03s] Complete run metadata saved: /Volumes/Amirali/hidden_states/experiments/baseline_v5_001/models/EleutherAI/gpt-neo-125m/datasets/goEmo/analysis/probes/matrix_runs/unified_v4_20260829_135913/complete_run_metadata.json
[probe + 1407.06s] ================================================================================================
[probe + 1407.06s] FINAL RESULT
[probe + 1407.06s] ================================================================================================
[probe + 1407.06s] Final best layer table:
          probe  layer_index  relative_layer_depth  probe_score_mean  test_macro_f1_mean  test_balanced_accuracy_mean  test_mcc_mean  selectivity_mean
linear_logistic            3                  0.25          0.342801            0.106718                     0.086905       0.306183          0.088891
   mlp_1_hidden            6                  0.50          0.325468            0.110448                     0.303571       0.074203          0.065524
   ml

Probing:   0%|          | 0/208 [00:00<?, ?fit/s]

In [ ]:
print(f"Matrix completed. Full results shape: {full_results.shape}")


In [ ]:
display(full_results.head())

In [ ]:
# Suppose we want metadata for the first row
sample_row = full_results.iloc[0]
metadata_path = sample_row["metadata_path"]
metadata = probe.load_complete_metadata(Path(metadata_path))
print(json.dumps(metadata, indent=2))

In [ ]:
if not full_results.empty:
    # Best layer per probe/model/dataset (highest test_macro_f1)
    best_per_probe = full_results.loc[full_results.groupby(["probe", "model", "dataset"])["test_macro_f1"].idxmax()]
    display_title("Best Layer per Probe (Macro-F1)")
    display(best_per_probe[["probe", "model", "dataset", "layer_index", "test_macro_f1", "probe_score"]])

    # Pivot table: model vs best macro-F1 per probe
    pivot_best = best_per_probe.pivot_table(index=["model", "dataset"], columns="probe", values="test_macro_f1")
    display_title("Best Macro-F1 Matrix")
    display(pivot_best.style.background_gradient(cmap='viridis', axis=None))

In [ ]:
output_plots_dir = Path("probe_plots")
output_plots_dir.mkdir(exist_ok=True)

# Use the plotting function from the probe module
probe.plot_full_dashboard(full_results, output_plots_dir)

In [ ]:
# Per model/dataset layer curves
for (model, dataset), group in full_results.groupby(["model", "dataset"]):
    plt.figure(figsize=(12, 6))
    for probe_name in group["probe"].unique():
        sub = group[group["probe"] == probe_name].sort_values("layer_index")
        plt.plot(sub["layer_index"], sub["test_macro_f1"], marker='o', label=probe_name)
    plt.title(f"{model} / {dataset} – Layer-wise Macro-F1")
    plt.xlabel("Layer index")
    plt.ylabel("Macro-F1")
    plt.legend()
    plt.grid(alpha=0.3)
    plt.tight_layout()
    plt.show()